# 01 — Data understanding

Phase 1 only. Observed NHSBSA community dispensing is not company sales. Synthetic fixtures used in tests are separate from observed analysis. Read the acquisition manifest for the active mode.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'config/dataset.json').exists(), 'Run from the project or notebooks directory'
df = pd.read_parquet(root / 'data/processed/monthly_prescribing.parquet')
source = pd.read_parquet(root / 'data/processed/source_rows.parquet')
manifest = json.loads((root / 'data/processed/acquisition_manifest.json').read_text(encoding='utf-8'))
quality = json.loads((root / 'outputs/validation/quality_report.json').read_text(encoding='utf-8'))

## Dataset overview and provenance
The extract is scoped to one chemical with multiple presentations. The raw API rows and region aggregation have different grains.

In [ ]:
display({'mode': manifest['mode'], 'source_shape': source.shape, 'monthly_shape': df.shape, 'source': manifest['dataset']['source_url'], 'monthly_key': manifest['processed_business_key']})
display(df.head(10))
display(source.head(5))

## Types, missingness and unique values
Blank optional supplier names mean unknown/not populated. Required key and measure missingness is an error.

In [ ]:
display(pd.DataFrame({'dtype': source.dtypes.astype(str), 'missing': source.isna().sum(), 'unique': source.nunique(dropna=False)}))
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)}))

## Descriptive statistics and time coverage

In [ ]:
display(df[['items', 'total_quantity', 'nic_gbp']].describe())
display({'first_month': df.month.min(), 'last_month': df.month.max(), 'months': df.month.nunique()})
coverage = df.groupby(['product_code', 'region_code', 'unit_of_measure']).month.agg(['min', 'max', 'nunique'])
display(coverage)

## Product/category distribution and geographic coverage
Quantity is not summed across presentations or units. Item counts are not unique patients.

In [ ]:
display(df.groupby(['chemical_code', 'chemical_name']).agg(presentations=('product_code','nunique'), items=('items','sum')))
display(df.groupby(['product_code','product_name']).agg(monthly_rows=('items','size'), items=('items','sum')))
display(df.groupby(['region_code','region_name']).agg(monthly_rows=('items','size'), items=('items','sum')))

## Initial descriptive view
No predictive model or forecast is fitted.

In [ ]:
monthly = df.groupby('month').items.sum()
ax = monthly.plot(figsize=(10,4), marker='.', title=f"{manifest['mode'].upper()}: selected atorvastatin prescription items")
ax.set_ylabel('Prescription items')
plt.tight_layout()
plt.show()

## Quality findings and business interpretation
Three-IQR outliers and absent series periods are review warnings. They are retained. Revenue arithmetic is not applicable because no observed revenue exists.

In [ ]:
display(pd.DataFrame(quality['checks']))
display(pd.DataFrame(quality.get('missing_series_periods', [])).head(20))
display(pd.DataFrame(quality.get('extremes', [])).head(20))
assert quality['passed'], 'Source or aggregation quality gate failed'
assert not df.duplicated(manifest['processed_business_key']).any()

## Supported and unsupported questions
Supports monthly dispensing volume, ingredient-cost trends, presentation mix and regional comparisons within the selected chemical. Cannot establish manufacturer revenue, India demand, promotion effects, physician engagement, profitability or whole-market share. Future forecasting requires sufficient complete series, time-based validation and explicit treatment of product lifecycle/geographic changes. See docs/INITIAL_DATA_ASSESSMENT.md.